# 08. Synthetic Data Recipe

- Parent issue: `#24`
- Status: `active`
- Summary: Implement synthetic data generation pipeline with solver-first / LLM-fallback teacher policy, quality filters, cost cap, smoke-run mode, and SHA-256 dataset fingerprinting. Produces SFTExample JSONL batches for SFT training.

## Audience and Why It Matters

Data generation owners, cost reviewers, and non-technical stakeholders evaluating risk and provenance.


## Decision / Hypothesis

Synthetic data should only be promoted when answers are verifiable, costs are bounded, and provenance is explicit.


## Environment and Reproduction

- Python: 3.11+
- Run from repo root: `jupyter notebook notebooks/08_synthetic_data_recipe.ipynb`
- Requires: `data/analysis/retry_candidates.jsonl` from Branch 1 (#22)
- Requires: `Verifier` implementation from Branch 2 (#23) for solver-first policy
- Cost warning: full generation run capped at $20; set `smoke=True` for initial inspection
- Companion registry entry: `docs/execution/NOTEBOOKS.md`

In [ ]:
from pathlib import Path
import platform

REPO_ROOT = Path.cwd()

print(f"Repo root: {REPO_ROOT}")
print(f"Python platform: {platform.platform()}")


In [ ]:
# Device detection: prefers CUDA, falls back to CPU gracefully.
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir() and (_candidate / 'notebooks').is_dir():
        REPO_ROOT = _candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.device import log_device_info
device = log_device_info()


## Method and Outputs

### Method

1. Load retry candidates from `data/analysis/retry_candidates.jsonl`.
2. Show `SyntheticConfig` and teacher policy (verifier-first → LLM fallback).
3. Show `QualityFilter` gates: dedupe, boxed, token length, provenance, category.
4. Run `--dry-run` to print token/cost estimate before committing to a full run.
5. Run smoke set (≤50 examples) — **HITL checkpoint**: human inspects output before authorizing full generation.
6. Verify fingerprint file written alongside output JSONL.

### Outputs produced

- `data/synthetic/<batch_id>.jsonl` — quality-filtered SFTExample rows
- `data/synthetic/<batch_id>.sha256` — SHA-256 fingerprint for reproducibility audit
- `configs/synthetic_prompts.yaml` — PromptCoT templates per category

In [ ]:
import sys
import json
from pathlib import Path

_cwd = Path.cwd()
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from src.data.synthetic import (
    SyntheticConfig,
    QualityFilter,
    generate_from_retry_candidates,
    write_sft_examples_jsonl,
)
from src.evaluation.trajectory import TrajectoryRow

RETRY_CANDIDATES_PATH = Path("data/analysis/retry_candidates.jsonl")
OUTPUT_DIR = Path("data/synthetic")

print(f"retry_candidates path: {RETRY_CANDIDATES_PATH}")
print(f"output_dir: {OUTPUT_DIR}")

# Load candidates (or use empty list if not yet produced by Branch 1)
candidates: list[TrajectoryRow] = []
if RETRY_CANDIDATES_PATH.exists():
    # TrajectoryRow JSONL is not directly deserializable without the full
    # EvalRecord constructor; in the real run, use the trajectory module.
    lines = [l for l in RETRY_CANDIDATES_PATH.read_text().splitlines() if l.strip()]
    print(f"Loaded {len(lines)} retry candidate line(s) from {RETRY_CANDIDATES_PATH}")
else:
    print("WARNING: retry_candidates.jsonl not found — using empty list for scaffold demo.")

print(f"\nCandidates available: {len(candidates)}")
import os

# Bind LLM teacher fn: real API if LLM_API_KEY is present, else stub.
_api_key = os.environ.get("LLM_API_KEY")
if _api_key:
    def _real_teacher(prompt: str) -> str:
        # Wire up actual LLM API call here (DeepSeek-R1, Claude, GPT-4, …)
        raise NotImplementedError("Replace _real_teacher with an API call before full generation.")
    llm_teacher_fn = _real_teacher
    print(f"LLM_API_KEY found (len={len(_api_key)}) — real LLM teacher will be used.")
else:
    llm_teacher_fn = lambda prompt: r"\boxed{STUB}"
    print("LLM_API_KEY not set — stub teacher active (returns \\boxed{STUB}).")


In [ ]:
# Show SyntheticConfig and teacher policy
config = SyntheticConfig(
    output_dir=OUTPUT_DIR,
    categories=["bit_manipulation", "math", "code", "science"],
    cost_cap_usd=20.0,
)

print("SyntheticConfig:")
print(f"  output_dir:    {config.output_dir}")
print(f"  categories:    {config.categories}")
print(f"  cost_cap_usd:  ${config.cost_cap_usd}")
print(f"  max_tokens:    {config.max_tokens}")

print("\nTeacher policy (in priority order):")
print("  1. Verifier.verify(pred, gold) — solver-first; uses Branch 2 Verifier Protocol")
print("  2. LLM teacher (DeepSeek-R1 → Claude → GPT-4 fallback)")
print("  answer=None from all → skip row")

print("\nQualityFilter gates:")
print("  - Dedupe: SHA-256 of example_id + category")
print("  - \\boxed{} present in completion")
print("  - Token length ≤ 8192")
print("  - Provenance: teacher, generated_at, source_run_id required")
print("  - Category in known taxonomy")
# Attach teacher function resolved above
config.llm_teacher_fn = llm_teacher_fn


In [ ]:
import time

# Generation loop — smoke run (capped at _SMOKE_LIMIT=50 examples).
print("Running smoke generation loop (smoke=True)…")
smoke_results = generate_from_retry_candidates(candidates, config, smoke=True)

output_path: str | None = None
candidates_taken = min(len(candidates), 50)
rejected_count = candidates_taken - len(smoke_results)

if smoke_results:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    batch_id = f"smoke_{int(time.time())}"
    out_file = OUTPUT_DIR / f"{batch_id}.jsonl"
    write_sft_examples_jsonl(smoke_results, out_file)
    # Store stem without extension so  output_path + '.sha256'  resolves correctly.
    output_path = str(OUTPUT_DIR / batch_id)
    print(f"Batch written : {out_file}  ({len(smoke_results)} example(s) passed quality filter)")
    print(f"Rejected      : {rejected_count}")
else:
    print("No examples generated — empty candidates list or all rejected by QualityFilter.")
    print("Provide data/analysis/retry_candidates.jsonl (Branch 1 #22) for real output.")

# ── HITL CHECKPOINT ────────────────────────────────────────────────────────
print()
print("=" * 60)
print("HITL CHECKPOINT")
print("Inspect smoke_results before authorising full generation.")
print("Remove smoke=True only after human review confirms quality.")
print("=" * 60)


In [ ]:
from pathlib import Path

# Fingerprint verification — reads the sibling .sha256 file written by write_sft_examples_jsonl.
if output_path is not None:
    fingerprint = Path(output_path + ".sha256").read_text().strip()
    print(f"Fingerprint file : {output_path}.sha256")
    print(f"SHA-256          : {fingerprint}")
else:
    fingerprint = "N/A"
    print("No output written — fingerprint verification skipped.")
    print("Re-run after providing retry_candidates.jsonl and a valid LLM_API_KEY (or stub).")


In [ ]:
# Summary table: rows generated / rejected / cost / fingerprint
cost_estimate_usd = len(smoke_results) * 0.01  # $0.01 placeholder per LLM call

rows = [
    ("Candidates fed (smoke cap)", min(len(candidates), 50)),
    ("Rows generated",             len(smoke_results)),
    ("Rows rejected",              rejected_count),
    ("Cost estimate (USD)",        f"${cost_estimate_usd:.4f}"),
    ("SHA-256 fingerprint",        (fingerprint[:24] + "…") if fingerprint != "N/A" else "N/A"),
]

col1 = max(len(r[0]) for r in rows) + 2
col2 = max(len(str(r[1])) for r in rows) + 2
header  = f"{'Metric':<{col1}} {'Value':>{col2}}"
divider = "-" * (col1 + col2 + 1)
print(header)
print(divider)
for label, value in rows:
    print(f"{label:<{col1}} {str(value):>{col2}}")


## Results / Open Risks

**Artifacts produced:**
- `src/data/synthetic.py` — generation pipeline with QualityFilter, teacher policy, cost cap
- `configs/synthetic_prompts.yaml` — PromptCoT templates per category
- `data/synthetic/<batch_id>.jsonl` + `.sha256` — generated SFTExample batches (after smoke approval)

**Open risks:**
- Requires `retry_candidates.jsonl` from Branch 1 (#22); scaffold runs with empty list.
- Real LLM teacher calls (`llm_teacher_fn`) must be configured before full generation — cost cap is enforced but API key setup is out of scope for this notebook.
- Smoke run HITL gate must be completed before authorizing full generation run.
- `Verifier.verify()` (Branch 2) must be instantiated and passed as `config.verifier` for solver-first policy.

## Sources

            - [Execution plan v0.2](docs/planning/plan_v0.2.md)
- [NVIDIA reasoning training blog](https://developer.nvidia.com/blog/train-a-reasoning-capable-llm-in-one-weekend-with-nvidia-nemo/)
- [SYNTHETIC-1](https://www.primeintellect.ai/blog/synthetic-1)
